## Document types

In [5]:
import pandas as pd
import altair as alt

from utils import pgp_csv_paths, chart_dir

documents = pd.read_csv(pgp_csv_paths["documents"])
total_documents = documents.shape[0]

# copy dataframe and replace unset type with "Unknown" so we can include it in reported type totals
docs_with_unknowns = documents.copy()
docs_with_unknowns["type"] = docs_with_unknowns.type.fillna("Unknown")
doctype_totals = docs_with_unknowns["type"].value_counts()

In [6]:
doctype_counts = doctype_totals.reset_index()
doctype_counts

,type,count
0,Letter,11281
1,Legal document,8096
2,List or table,5595
3,Unknown,4005
4,Literary text,2493
5,State document,2209
6,Paraliterary text,1196
7,Credit instrument or private receipt,568
8,Legal query or responsum,407
9,Inscription,5


In [7]:
# omit Inscription, currently only 5 items
doctype_chart = alt.Chart(doctype_counts[doctype_counts['count'] > 10]).mark_bar().encode(
    y=alt.Y('type', title="Document Type"),
    x=alt.X('count', title="Number of documents")
).properties(
    # title="Documents by type",
    width=600
)
doctype_chart.save(f"{chart_dir}/document_types.pdf")
doctype_chart

alt.Chart(...)

### Document descriptions

What is the typical length of document descriptions?

In [8]:
documents['description_num_chars'] = documents.description.apply(lambda x: len(x.strip()) if pd.notna(x) else 0)
documents['description_num_tokens'] = documents.description.apply(lambda x: len(x.strip().split()) if pd.notna(x) else 0)
documents[['pgpid', 'description', 'description_num_chars', 'description_num_tokens']].head()

,pgpid,description,description_num_chars,description_num_tokens
0,444,Fragment of a draft letter in the name of the ...,290,51
1,445,Fragment of a legal document in which the sign...,81,14
2,446,"A writ of qiddushin (betrothal), Tyre, ca. 101...",53,8
3,447,"Legal document. According to Ashtor, ""an agree...",460,74
4,448,"Letter from Mufaḍḍal, probably in Fustat, to A...",1678,269


In [9]:
documents[documents.description_num_chars == 0]

,pgpid,url,iiif_urls,fragment_urls,shelfmark,multifragment,side,region,type,tags,...,inferred_date_notes,initial_entry,last_modified,input_by,library,collection,has_transcription,has_translation,description_num_chars,description_num_tokens
29776,35193,https://geniza.princeton.edu/documents/35193/,NaN,NaN,TS NS 31.26,NaN,NaN,NaN,NaN,NaN,...,NaN,2022-04-07 17:33:25.758174+00:00,2022-04-07 17:33:25.451399+00:00,Amir Ashur,CUL,"CUL, T-S",N,N,0,0


In [10]:
# langdetect is not useful here
# could potentially analyze based on unicode script range
# i.e., what percentage have hebrew / arabic terms (usually names?)
# but this is probably not important